# DOC TO CSV

In [1]:
import os
import pandas as pd
from docx import Document
import json

def extract_json_from_docx(docx_path):
    document = Document(docx_path)
    json_text = ""
    
    for paragraph in document.paragraphs:
        json_text += paragraph.text.strip()
    
    try:
        # Parse JSON text into a Python object
        return json.loads(json_text)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON in {docx_path}: {e}")
        return None

def normalize_flight_data(json_data):
    data = []
    for record in json_data:
        flat_record = {
            "type": record.get("type"),
            "status": record.get("status"),
            # Departure details
            "departure_iata": record.get("departure", {}).get("iataCode"),
            "departure_icao": record.get("departure", {}).get("icaoCode"),
            "departure_terminal": record.get("departure", {}).get("terminal"),
            "departure_scheduled": record.get("departure", {}).get("scheduledTime"),
            "departure_estimated": record.get("departure", {}).get("estimatedTime"),
            "departure_actual": record.get("departure", {}).get("actualTime"),
            "departure_runway_estimated": record.get("departure", {}).get("estimatedRunway"),
            "departure_runway_actual": record.get("departure", {}).get("actualRunway"),
            # Arrival details
            "arrival_iata": record.get("arrival", {}).get("iataCode"),
            "arrival_icao": record.get("arrival", {}).get("icaoCode"),
            "arrival_terminal": record.get("arrival", {}).get("terminal"),
            "arrival_scheduled": record.get("arrival", {}).get("scheduledTime"),
            "arrival_estimated": record.get("arrival", {}).get("estimatedTime"),
            # Airline details
            "airline_name": record.get("airline", {}).get("name"),
            "airline_iata": record.get("airline", {}).get("iataCode"),
            "airline_icao": record.get("airline", {}).get("icaoCode"),
            # Flight details
            "flight_number": record.get("flight", {}).get("number"),
            "flight_iata": record.get("flight", {}).get("iataNumber"),
            "flight_icao": record.get("flight", {}).get("icaoNumber"),
        }
        data.append(flat_record)
    return pd.DataFrame(data)

def process_docx_directory(directory_path):
    """
    Process all DOCX files in a directory and convert them into a single DataFrame.
    """
    all_data = pd.DataFrame()  # Empty DataFrame to store all data
    for file_name in os.listdir(directory_path):
        if file_name.endswith(".docx"):
            file_path = os.path.join(directory_path, file_name)
            print(f"Processing {file_name}...")
            
            json_data = extract_json_from_docx(file_path)
            if json_data:
                df = normalize_flight_data(json_data)
                all_data = pd.concat([all_data, df], ignore_index=True)

    return all_data

directory_path = "Train"  

combined_df = process_docx_directory(directory_path)

if not combined_df.empty:
    print(combined_df.head()) 
    
    csv_path = "combined_flight_data.csv"  # Update this path if needed
    combined_df.to_csv(csv_path, index=False)
    print(f"CSV saved at {csv_path}")
else:
    print("No data found in DOCX files.")


Processing 1.docx...
Processing 10.docx...
Processing 11.docx...
Processing 12.docx...
Processing 13.docx...
Processing 14.docx...
Processing 15.docx...
Processing 16.docx...
Processing 17.docx...
Processing 18.docx...
Processing 19.docx...
Processing 2.docx...
Processing 20.docx...
Processing 21.docx...
Processing 22.docx...
Processing 23.docx...
Processing 24.docx...
Processing 25.docx...
Processing 26.docx...
Processing 27.docx...
Processing 28.docx...
Processing 29.docx...
Processing 3.docx...
Processing 30.docx...
Processing 31.docx...
Processing 32.docx...
Processing 33.docx...
Processing 34.docx...
Processing 35.docx...
Processing 36.docx...
Processing 37.docx...
Processing 39.docx...
Processing 4.docx...
Processing 40.docx...
Processing 41.docx...
Processing 42.docx...
Processing 43.docx...
Processing 44.docx...
Processing 45.docx...
Processing 46.docx...
Processing 47.docx...
Processing 48.docx...
Processing 49.docx...
Processing 5.docx...
Processing 50.docx...
Processing 51.d

# XLSX TO CSV

In [ ]:
import pandas as pd
import numpy as np

file_path = "1.xlsx"  
data = pd.ExcelFile(file_path)

sheet1_data = data.parse("Sheet1")

raw_data = sheet1_data.iloc[0]

split_data = {col: raw_data[col].split() for col in raw_data.index}

time_column = split_data["Time"]

cleaned_columns = {}
for col, values in split_data.items():
    if col == "Time":
        continue
    try:
        reshaped_values = np.array(values).reshape(-1, 3)
        cleaned_columns[f"{col} Max"] = reshaped_values[:, 0]
        cleaned_columns[f"{col} Avg"] = reshaped_values[:, 1]
        cleaned_columns[f"{col} Min"] = reshaped_values[:, 2]
    except ValueError:
        cleaned_columns[col] = values

cleaned_data = pd.DataFrame({
    "Time": time_column,
    **cleaned_columns
})

for col in cleaned_data.columns[1:]:
    cleaned_data[col] = pd.to_numeric(cleaned_data[col], errors="coerce")

cleaned_data.to_csv("cleaned_data.csv", index=False)

print("Data cleaned and saved to 'cleaned_data.csv'")

In [24]:
import pandas as pd
import numpy as np

def clean_and_reshape_data(data):
    cleaned_dfs = []
    for col, values in data.items():
        if col == "Time":
            cleaned_columns["Time"] = values
            continue

        numeric_values = pd.to_numeric(values, errors='coerce')

        df_col = pd.DataFrame({'values': numeric_values})

        df_reshaped = df_col.values.reshape(-1, 3)
        df_reshaped = pd.DataFrame(df_reshaped, columns=[f"{col} Max", f"{col} Avg", f"{col} Min"])

        cleaned_dfs.append(df_reshaped)

    return pd.concat(cleaned_dfs, axis=1)

file_path = "Weather/8.xlsx" 
data = pd.ExcelFile(file_path)

sheet1_data = data.parse("Sheet1")
raw_data = sheet1_data.iloc[0]

split_data = {col: raw_data[col].split() for col in raw_data.index}

cleaned_data = clean_and_reshape_data(split_data)

cleaned_data.to_csv("8_cleaned.csv", index=False)

print("Data cleaned and saved to '8_cleaned.csv'")

Data cleaned and saved to '8_cleaned.csv'


In [28]:
import pandas as pd
import os

folder_path = 'csv'

csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

dataframes = []
for file in csv_files:
    file_path = os.path.join(folder_path, file)
    df = pd.read_csv(file_path)
    dataframes.append(df)
combined_df = pd.concat(dataframes, ignore_index=True)
combined_df.to_csv('combined_file.csv', index=False)

print("All CSV files have been combined into 'combined_file.csv'.")


All CSV files have been combined into 'combined_file.csv'.


In [29]:
file1=pd.read_csv('combined_flight_data.csv')

In [31]:
file1.sample(10)

,type,status,departure_iata,departure_icao,departure_terminal,departure_scheduled,departure_estimated,departure_actual,departure_runway_estimated,departure_runway_actual,...,arrival_icao,arrival_terminal,arrival_scheduled,arrival_estimated,airline_name,airline_iata,airline_icao,flight_number,flight_iata,flight_icao
3319,departure,active,lhe,opla,m,2024-02-06t02:55:00.000,2024-02-06t02:55:00.000,2024-02-06t03:02:00.000,2024-02-06t03:02:00.000,2024-02-06t03:02:00.000,...,othh,NaN,2024-02-06t05:05:00.000,2024-02-06t04:48:00.000,rwandair,wb,rwd,1569,wb1569,rwd1569
24563,departure,active,lhe,opla,m,2023-09-08t09:55:00.000,2023-09-08t09:55:00.000,2023-09-08t09:59:00.000,2023-09-08t09:59:00.000,2023-09-08t09:59:00.000,...,othh,NaN,2023-09-08t11:55:00.000,2023-09-08t11:04:00.000,royal jordanian,rj,rja,3836,rj3836,rja3836
41427,departure,active,lhe,opla,m,2023-10-15t08:00:00.000,NaN,2023-10-15t07:57:00.000,2023-10-15t07:57:00.000,2023-10-15t07:57:00.000,...,oejn,n,2023-10-15t11:30:00.000,2023-10-15t10:55:00.000,airsial,pf,sif,716,pf716,sif716
48686,departure,cancelled,lhe,opla,m,2023-10-22t11:00:00.000,NaN,NaN,NaN,NaN,...,opkc,m,2023-10-22t12:45:00.000,NaN,pakistan international airlines,pk,pia,303,pk303,pia303
4333,departure,active,lhe,opla,main,2024-02-17t09:35:00.000,2024-02-17t10:02:00.000,NaN,NaN,NaN,...,omdb,3,2024-02-17t12:05:00.000,NaN,emirates,ek,uae,625,ek625,uae625
4061,departure,active,lhe,opla,m,2024-02-23t11:40:00.000,NaN,2024-02-23t11:50:00.000,2024-02-23t11:50:00.000,2024-02-23t11:50:00.000,...,oejn,1,2024-02-23t15:25:00.000,2024-02-23t15:18:00.000,kenya airways,kq,kqa,5783,kq5783,kqa5783
28110,departure,active,khi,opkc,NaN,2024-04-25t08:30:00.000,NaN,2024-04-25t08:43:00.000,2024-04-25t08:43:00.000,2024-04-25t08:43:00.000,...,othh,NaN,2024-04-25t08:55:00.000,2024-04-25t09:00:00.000,qatar airways,qr,qtr,8681,qr8681,qtr8681
9514,departure,active,lhe,opla,m,2024-06-18t03:10:00.000,2024-06-18t03:10:00.000,2024-06-18t03:14:00.000,2024-06-18t03:14:00.000,2024-06-18t03:14:00.000,...,othh,NaN,2024-06-18t04:55:00.000,2024-06-18t04:31:00.000,iberia,ib,ibe,7913,ib7913,ibe7913
45580,departure,active,isb,opis,NaN,2024-03-17t03:10:00.000,2024-03-17t03:10:00.000,2024-03-17t03:25:00.000,2024-03-17t03:25:00.000,2024-03-17t03:25:00.000,...,othh,NaN,2024-03-17t05:20:00.000,2024-03-17t05:08:00.000,american airlines,aa,aal,8311,aa8311,aal8311
47372,departure,active,isb,opis,NaN,2024-05-09t05:45:00.000,2024-05-09t05:45:00.000,2024-05-09t05:52:00.000,2024-05-09t05:52:00.000,2024-05-09t05:52:00.000,...,omdb,2,2024-05-09t08:15:00.000,2024-05-09t07:50:00.000,emirates,ek,uae,2115,ek2115,uae2115
